# Visualize a ladybug database <a target="_blank" href="https://colab.research.google.com/github/yWorks/yfiles-jupyter-graphs-for-kuzu/blob/kuzu/examples/ladybug.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook shows how to visualize a LadybugDB database with `yfiles-jupyter-graphs-for-kuzu`.

## Install required dependencies

In [ ]:
%pip install yfiles_jupyter_graphs_for_kuzu --quiet
%pip install ladybug --quiet

You can also open this notebook in Google Colab when Google Colab's custom widget manager is enabled:

In [ ]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

## Create a graph

For demonstration purpose we create a network with fictional relationships.

In [ ]:
import ladybug as lb

# Open a new in-memory database
db = lb.Database(":memory:")
conn = lb.Connection(db)

# Create the person table
conn.execute("""
    CREATE NODE TABLE Person(
        name STRING PRIMARY KEY,
        alias STRING
    )
""")

# Create several relationship types.
# Relationship direction is meaningful:
# - REVIEWED: reviewer -> author
# - RECOMMENDED: recommender -> candidate
# - COLLABORATED: project lead -> collaborator
conn.execute("CREATE REL TABLE MENTORED(FROM Person TO Person)")
conn.execute("CREATE REL TABLE COLLABORATED(FROM Person TO Person)")
conn.execute("CREATE REL TABLE COAUTHORED(FROM Person TO Person)")
conn.execute("CREATE REL TABLE REVIEWED(FROM Person TO Person)")
conn.execute("CREATE REL TABLE RECOMMENDED(FROM Person TO Person)")

# Insert fictional people and their professional relationships
conn.execute("""
    // People
    MERGE (a:Person {
        name: "Mira Vale",
        alias: "Dr. Mira Vale"
    })
    MERGE (b:Person {
        name: "Theo Arden",
        alias: "Dr. Theo Arden"
    })
    MERGE (c:Person {
        name: "Leila Noor",
        alias: "Dr. Leila Noor"
    })
    MERGE (d:Person {
        name: "Anton Reyes",
        alias: "Dr. Anton Reyes"
    })
    MERGE (e:Person {
        name: "Priya Sen",
        alias: "Dr. Priya Sen"
    })
    MERGE (f:Person {
        name: "Rowan Kepler",
        alias: "Dr. Rowan Kepler"
    })
    MERGE (g:Person {
        name: "Hana Ito",
        alias: "Dr. Hana Ito"
    })
    MERGE (h:Person {
        name: "Elias Ward",
        alias: "Dr. Elias Ward"
    })
    MERGE (i:Person {
        name: "Sofia Marin",
        alias: "Dr. Sofia Marin"
    })
    MERGE (j:Person {
        name: "Daniel Okafor",
        alias: "Dr. Daniel Okafor"
    })

    // Mentorship
    MERGE (a)-[:MENTORED]->(b)
    MERGE (a)-[:MENTORED]->(c)
    MERGE (b)-[:MENTORED]->(e)
    MERGE (c)-[:MENTORED]->(d)
    MERGE (c)-[:MENTORED]->(g)
    MERGE (d)-[:MENTORED]->(f)
    MERGE (e)-[:MENTORED]->(i)
    MERGE (g)-[:MENTORED]->(h)
    MERGE (h)-[:MENTORED]->(j)

    // Research collaborations
    MERGE (a)-[:COLLABORATED]->(c)
    MERGE (b)-[:COLLABORATED]->(d)
    MERGE (e)-[:COLLABORATED]->(g)
    MERGE (f)-[:COLLABORATED]->(h)
    MERGE (i)-[:COLLABORATED]->(j)

    // Co-authorship relationships
    MERGE (a)-[:COAUTHORED]->(b)
    MERGE (b)-[:COAUTHORED]->(e)
    MERGE (c)-[:COAUTHORED]->(d)
    MERGE (d)-[:COAUTHORED]->(f)
    MERGE (g)-[:COAUTHORED]->(h)
    MERGE (h)-[:COAUTHORED]->(j)

    // Peer review: reviewer -> author
    MERGE (c)-[:REVIEWED]->(b)
    MERGE (d)-[:REVIEWED]->(e)
    MERGE (f)-[:REVIEWED]->(g)
    MERGE (i)-[:REVIEWED]->(h)

    // Professional recommendations
    MERGE (a)-[:RECOMMENDED]->(e)
    MERGE (b)-[:RECOMMENDED]->(f)
    MERGE (c)-[:RECOMMENDED]->(i)
    MERGE (g)-[:RECOMMENDED]->(j)
""")

# Database visualization

Easily visualize the result of a query.

In [ ]:
from yfiles_jupyter_graphs_for_kuzu import KuzuGraphWidget
g = KuzuGraphWidget(conn)
# Display the entire connected graph
g.show_cypher("""
    MATCH (source)-[relationship]->(target)
    RETURN source, relationship, target
    LIMIT 100
""", 'hierarchical')

# Data-driven visualization

You can specify node and relationship configurations to change visualization properties based on the graph items.

In [ ]:
g.add_node_configuration(
    "Person",
    styles={"color": "#4CAF50", "shape": "squircle"}
)
g.add_relationship_configuration(
    "MENTORED",
    color="red",
    thickness_factor=2
)                                 
g.show_cypher("""
    MATCH (source)-[relationship]->(target)
    RETURN source, relationship, target
    LIMIT 100
""", 'hierarchical')